# Metropolis Monte Carlo of dipolar Lennard-Jones particles

*Utrecht University Molecular Modelling courses from the [Bonvin lab](https://bonvinlab.org).*

This notebook is the **dipolar** variant of the Metropolis Monte Carlo notebook (`LJ-ELEC_MMC`).
Every particle carries an **oriented point dipole** $\vec m$, drawn as a green arrow. **By default
the particles are electrically neutral**, so the only interactions are Lennard-Jones plus the
**dipole–dipole** coupling — the structure that forms is therefore *purely* dipolar. A new Monte
Carlo move, a **rotation** of a dipole, lets the orientations equilibrate alongside the positions.
Watching the run, the initially random arrows tend to **order** — lining up head-to-tail into
chains and rings — the hallmark of dipolar coupling.

> **Charges are optional.** With the default `qat = 0` the particles carry no net charge and the
> Coulomb term is switched off. Set `qat = Radius` in the **Parameters** cell to give every particle
> a monopole charge as well, and watch the (much stronger, longer-ranged) Coulomb interaction
> compete with — and partly scramble — the dipolar ordering.

## Theory in brief

### Lennard-Jones and (optional) Coulomb
As before, from the squared distance with $Z = (r_{min}^2/r^2)^3 = (r_{min}/r)^6$:

$$E_{LJ} = \varepsilon\, Z(Z-1), \qquad E_{Coul} = \frac{q_a q_b}{\epsilon_r\, r}.$$

With the default neutral particles ($q_a = q_b = 0$) the Coulomb term simply vanishes; it only
enters if you switch charges on with `qat > 0`.

### Dipole–dipole interaction
Each particle has a 2D dipole moment $\vec m_i = mm\,(\cos\theta_i, \sin\theta_i)$ of fixed
magnitude `mm` and orientation $\theta_i$. Two dipoles separated by $\vec r$ (with unit vector
$\hat r$) interact with energy

$$E_{dip} = \frac{1}{r^3}\Big[\,\vec m_1\!\cdot\!\vec m_2 - 3(\vec m_1\!\cdot\!\hat r)(\vec m_2\!\cdot\!\hat r)\,\Big].$$

This is **orientation-dependent**: two dipoles aligned **head-to-tail** along $\vec r$ attract
($E<0$), while **side-by-side** parallel dipoles repel ($E>0$). Because it falls off as $1/r^3$ it
is short-ranged, and it *couples the orientations to the positions* — which is why the dipoles
tend to order into head-to-tail chains as the system relaxes.

### The Metropolis moves
The acceptance rule is unchanged — a trial move is accepted with
$P_{acc} = \min(1, e^{-\Delta E/k_B T})$ — but there are now **three** kinds of trial move:

* a **displacement** of one coordinate of a random atom (probability $1-\texttt{frac\_swap}-\texttt{frac\_rot}$);
* a **charge swap** — two atoms exchange positions (probability `frac_swap`);
* a **dipole rotation** — a random atom's orientation $\theta$ is changed (probability `frac_rot`).

$\Delta E$ is always evaluated from the **full** energy (LJ + Coulomb + dipole), so every move type
feels the dipolar coupling.

## 1. Imports

The numerical core uses only the Python **standard library** (`math`, `random`), so it runs
on a bare Python install. **matplotlib** is the one third-party dependency — it draws the
static figures and the trajectory animation (embedded inline as interactive HTML via
`jshtml`). The cell below first **installs matplotlib if it is missing** (handy on Google
Colab), then imports everything; `%matplotlib inline` renders figures inside the notebook.

In [ ]:
# --- Install required packages if missing (e.g. on Google Colab) ---
import importlib.util, subprocess, sys

for pkg in ["matplotlib"]:
    if importlib.util.find_spec(pkg) is None:
        print(f"Installing {pkg} ...")
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", pkg], check=True)
    else:
        print(f"{pkg} already available")

from math import sqrt, exp, pi, cos, sin

import matplotlib.pyplot as plt
from matplotlib.patches import Circle
from matplotlib.animation import FuncAnimation
from matplotlib import rc

from random import random, randint, seed

# Show animations inline
rc('animation', html='jshtml')
%matplotlib inline

## 2. Helper functions

Small utilities used throughout:

* `dist` / `dist2` — Euclidean distance and its square.
* `SignR(a, b)` — returns `a` with the sign of `b`. It implements the **minimum-image
  convention** (`tmp - SignR(halfbox, tmp-halfbox) - SignR(halfbox, tmp+halfbox)` wraps a
  coordinate difference into $[-\tfrac{box}{2}, +\tfrac{box}{2}]$), i.e. periodic boundary
  conditions. The same trick keeps a displaced atom inside the box.
* `charge_color` — colours a particle by charge (neutral particles are drawn grey).
* `local_order` — the **dipole-order parameter** used below: the average of
  $\cos(\theta_i-\theta_j)$ over neighbouring pairs (closer than $1.5\,r_{min}$). It is ~0 for
  random orientations and grows as neighbouring dipoles align head-to-tail (see §8 for why a
  *local* measure — not the net polarization — is the right choice here).

In [ ]:
### distance ###
def dist(A, B):
    return sqrt((A[0]-B[0])**2 + (A[1]-B[1])**2)

### squared distance ###
def dist2(A, B):
    return (A[0]-B[0])**2 + (A[1]-B[1])**2

### change sign ###
def SignR(a, b):
    if b > 0:
        return a
    else:
        return -a

### colour particles based on charge (neutral particles are grey) ###
def charge_color(charge, qat):
    if qat == 0:
        return "#999999"   # neutral
    if charge == qat:
        return "#FFFFFF"   # positive
    else:
        return "#333333"   # negative

### local dipole-order parameter: mean cos(theta_i - theta_j) over neighbour pairs ###
def local_order(coord, orient, boxdim, cutoff):
    cutoffsq = cutoff * cutoff
    tot = 0.0
    cnt = 0
    for i in range(len(coord)-1):
        for j in range(i+1, len(coord)):
            distsquare = 0
            for k in range(2):
                tmp = coord[j][k] - coord[i][k]
                halfbox = boxdim[k]/2
                tmp = tmp - SignR(halfbox, tmp-halfbox) - SignR(halfbox, tmp+halfbox)
                distsquare += tmp**2
            if distsquare < cutoffsq:
                tot += cos(orient[i] - orient[j])
                cnt += 1
    return tot/cnt if cnt else 0.0

## 3. Energy functions

Total energy = Lennard-Jones + Coulomb + **dipole–dipole**, summed over all pairs with the
**nearest image** convention. `DipoleEne` implements the 2D dipole–dipole formula above using the
nearest-image vector $\vec r_{12}$ that `Calc_EneDip` already builds. `Calc_EneDip` returns the
total energy together with its three components, so we can watch them separately.

In [ ]:
# LJ energy from the squared distance
def LJ2(distsquare, epsilon, rmin_exp6):
    Z = (1/distsquare)**3 * rmin_exp6
    return epsilon * Z * (Z - 1)

# classical Coulomb from the squared distance
def Coulomb2(r, dielec, qa, qb):
    return qa*qb / (dielec*sqrt(r))

# 2D dipole-dipole energy:  U = [ m1.m2 - 3 (m1.rhat)(m2.rhat) ] / r^3
def DipoleEne(m1, m2, r12):
    l = sqrt(r12[0]**2 + r12[1]**2)
    fact0 = 1.0 / (l**3)
    fact1 = m1[0]*m2[0] + m1[1]*m2[1]          # m1 . m2
    g0 = (r12[0]*m1[0] + r12[1]*m1[1]) / l     # m1 . rhat
    g1 = (r12[0]*m2[0] + r12[1]*m2[1]) / l     # m2 . rhat
    return fact0 * (fact1 - 3.0*g0*g1)

# Total energy (LJ + Coulomb + dipole-dipole) with periodic boundary conditions.
# Returns (Etot, ELJ, ECoul, EDip).
def Calc_EneDip(coord, orient, epsilon, rmin, dielec, cutoffsquare, boxdim, mm, elec=1):
    ELJ = 0.0
    ECoul = 0.0
    EDip = 0.0
    rmin_exp6 = rmin**6
    for i in range(len(coord)-1):
        mu1 = [mm*cos(orient[i]), mm*sin(orient[i])]
        for j in range(i+1, len(coord)):
            mu2 = [mm*cos(orient[j]), mm*sin(orient[j])]
            # squared distance and nearest-image vector r12
            r12 = []
            distsquare = 0
            for k in range(2):
                tmp = coord[j][k] - coord[i][k]
                halfbox = boxdim[k]/2
                tmp = tmp - SignR(halfbox, tmp-halfbox) - SignR(halfbox, tmp+halfbox)
                r12.append(tmp)
                distsquare += tmp**2
            if distsquare < cutoffsquare:
                qa = coord[i][2]
                qb = coord[j][2]
                ELJ += LJ2(distsquare, epsilon, rmin_exp6)
                if elec:
                    ECoul += Coulomb2(distsquare, dielec, qa, qb)
                EDip += DipoleEne(mu1, mu2, r12)
    return ELJ + ECoul + EDip, ELJ, ECoul, EDip

## 4. The Metropolis Monte Carlo sampler

The headless loop over `nsteps` trial moves. Each step picks a random atom and, with the
respective probabilities, proposes a **displacement**, a **charge swap**, or a **dipole
rotation**; it then evaluates the change in the **full** energy (so the dipole term is always
included) and applies the Metropolis test. Rejected moves are undone. The running energy (and its
LJ/Coulomb/dipole parts), the acceptance ratio, the **dipole-order parameter**, and snapshots of
both positions and orientations are recorded at every step.

In [ ]:
def run_mc(Atom_Coord, Atom_Orient):
    """Headless Metropolis MC with translation, swap and dipole-rotation moves."""
    coord  = [list(a) for a in Atom_Coord]
    orient = list(Atom_Orient)

    Ene, ELJ, ECoul, EDip = Calc_EneDip(coord, orient, Epsilon, Rmin, Dielec, CutOffSquare, BoxDim, mm)
    traj      = [[list(a) for a in coord]]
    otraj     = [list(orient)]
    E_hist    = [Ene]
    Elj_hist  = [ELJ]
    Ecoul_hist = [ECoul]
    Edip_hist = [EDip]
    acc_hist  = [1.0]
    order_hist = [local_order(coord, orient, BoxDim, 1.5*Rmin)]
    move_hist = ["init"]

    Accepted = 0
    frac_simple_move = 1 - frac_swap - frac_rot
    kT = cstboltz * Temperature

    print("step %6d  E= %.2f   order= %.3f" % (0, Ene, order_hist[0]))

    for step in range(1, nsteps+1):
        ra = randint(0, len(coord)-1)
        Xold, Yold = coord[ra][0], coord[ra][1]
        Oold = orient[ra]
        xx = random()

        if xx < frac_simple_move:
            movetype = "move"
            rc = randint(0, 1)
            z = coord[ra][rc] + (2*random() - 1)*deltaRmax
            halfbox = BoxDim[rc]/2
            coord[ra][rc] = z - SignR(halfbox, z) - SignR(halfbox, z - BoxDim[rc])
        elif (xx - frac_simple_move) < frac_swap:
            movetype = "swap"
            ra2 = randint(0, len(coord)-1)
            Xb, Yb = coord[ra2][0], coord[ra2][1]
            coord[ra][0],  coord[ra][1]  = Xb, Yb
            coord[ra2][0], coord[ra2][1] = Xold, Yold
        else:
            movetype = "rotate"
            orient[ra] = Oold + random()*2.0*pi

        En, LJn, Con, Dn = Calc_EneDip(coord, orient, Epsilon, Rmin, Dielec, CutOffSquare, BoxDim, mm)
        dE = En - Ene
        if dE < 0.0 or random() < exp(-dE/kT):
            Ene, ELJ, ECoul, EDip = En, LJn, Con, Dn
            Accepted += 1
        else:
            if movetype == "move":
                coord[ra][rc] = Xold if rc == 0 else Yold
            elif movetype == "swap":
                coord[ra][0],  coord[ra][1]  = Xold, Yold
                coord[ra2][0], coord[ra2][1] = Xb, Yb
            else:
                orient[ra] = Oold

        traj.append([list(a) for a in coord])
        otraj.append(list(orient))
        E_hist.append(Ene); Elj_hist.append(ELJ); Ecoul_hist.append(ECoul); Edip_hist.append(EDip)
        acc_hist.append(Accepted/step)
        order_hist.append(local_order(coord, orient, BoxDim, 1.5*Rmin))
        move_hist.append(movetype)

        if step % 1000 == 0:
            print("step %6d  E= %.2f   order= %.3f   P(accept)= %.3f"
                  % (step, Ene, order_hist[-1], Accepted/step))

    print("\nFinal acceptance ratio: %.3f   final order parameter: %.3f" % (Accepted/nsteps, order_hist[-1]))
    return traj, otraj, E_hist, Elj_hist, Ecoul_hist, Edip_hist, acc_hist, order_hist, move_hist

## 5. Parameters

This variant of `LJ-ELEC_MMC` adds the **dipole** controls `mm` (dipole strength) and `frac_rot`
(fraction of rotation moves). Change any of them and re-run **this cell together with the
Initialisation and Run cells just below**.

> **Neutral by default.** To show the dipole physics *on its own*, the particles are **neutral**
> here (`qat = 0`), so the monopole Coulomb term is switched off and any structure that forms is due
> to the dipoles alone. Set `qat = Radius` to put charges back and watch Coulomb compete with the
> dipolar ordering. With identical neutral particles a charge *swap* does nothing, so `frac_swap = 0`
> and those moves are given to rotations instead.
>
> **Note on `mm`.** The dipole energy scales like $mm^2/r^3$, so the dipole strength must be chosen
> to match the rest of the energy scale. At these parameters `mm = 1000` makes the dipoles order
> without freezing the system; much larger values reject almost every move.

### System and its properties

| Parameter | Meaning | Typical value / range |
|---|---|---|
| `nAtoms` | number of particles | 2–40 |
| `Radius` | particle radius | 10–40 |
| `Rmin` | position of the LJ energy minimum | `2 * Radius` |
| `BoxDim` | box dimensions (periodic) | `[500, 500]` |
| `Epsilon` | LJ well depth | 1–100 |
| `Dielec` | dielectric constant | 1 (vacuum) – 80 (water) |
| `qat` | absolute charge per atom | **0 (neutral)**; `Radius` to add charges |
| `frac_neg` | fraction of negative charges (only matters if `qat` > 0) | 0–1 |
| `CutOff` | non-bonded cutoff distance | 250 |

### Monte Carlo and dipole controls

| Parameter | Meaning | Typical value / range |
|---|---|---|
| `deltaRmax` | maximum displacement of a trial move | 30 |
| `frac_swap` | fraction of trial moves that are charge swaps | 0 (neutral) |
| `frac_rot` | fraction of trial moves that are dipole rotations | 0.35 |
| `mm` | dipole strength (energy scales as $mm^2/r^3$) | 1000 |
| `Temperature` | temperature `T` in the Boltzmann factor | 300 |
| `cstboltz` | Boltzmann constant (energy scale of $k_B T$) | fixed |
| `Seed` | random-number seed | 100 |
| `nsteps` | number of Monte Carlo trial moves | 15000 |

In [ ]:
nAtoms  = 40              # number of particles
Radius  = 25.0            # particle radius
Rmin    = 2 * Radius      # distance at which the LJ energy is minimal
BoxDim  = [500, 500]      # box dimensions
Epsilon = 20.0            # LJ well depth
Dielec  = 1.0             # dielectric constant
qat     = 0.0             # atom absolute charge (0 = neutral; set to Radius to add monopole charges)
frac_neg = 0.5            # fraction of negative charges (only matters if qat > 0)
OverlapFr = 0.0           # fraction of overlap allowed when placing atoms
CutOff  = 250             # non-bonded cutoff
CutOffSquare = CutOff**2

# --- Monte Carlo + dipole controls ---
deltaRmax   = 30.0             # maximum displacement of a trial move
frac_swap   = 0.0              # fraction of charge-swap moves (0: pointless for identical neutral particles)
frac_rot    = 0.35             # fraction of trial moves that are dipole rotations
mm          = 1000.0           # dipole strength (energy ~ mm^2 / r^3)
Temperature = 300.0           # temperature in the Boltzmann acceptance factor
cstboltz    = 8.3502e-03       # Boltzmann constant (J/mol/K)
Seed        = 100             # random number seed
nsteps      = 15000           # number of Monte Carlo trial moves

## 6. Initialisation

Generate random, non-overlapping starting positions, assign charges (a fraction `frac_neg`
negative), and give each particle a **random dipole orientation** in $[0, 2\pi)$. `seed(Seed)`
makes the run reproducible.

In [ ]:
import sys

### random, non-overlapping coordinates + random dipole orientations ###
def InitConf(n, dim, radius, qat, frac_neg):
    seed(Seed)
    print("Initializing box, please wait...")
    tmp_coord = []
    tmp_orient = []
    i = 0
    ntrial = 0
    nneg = int(float(n) * frac_neg)
    npos = n - nneg

    # first atom (no overlap check)
    x = random()*(dim[0]-radius) + radius
    y = random()*(dim[1]-radius) + radius
    charge = -qat
    if npos == n:
        charge = qat
    i += 1
    tmp_coord.append([x, y, charge])
    tmp_orient.append(2.0*random()*pi)

    # remaining negative charges
    while i < nneg:
        x = random()*(dim[0]-radius) + radius
        y = random()*(dim[1]-radius) + radius
        angle = 2.0*random()*pi
        OVERLAP = 1
        for j in range(i):
            if dist(tmp_coord[j], [x, y]) < (1-OverlapFr)*2*radius:
                OVERLAP = 0
        if OVERLAP:
            tmp_coord.append([x, y, -qat])
            tmp_orient.append(angle)
            i += 1
        ntrial += 1
        if ntrial > 100000:
            print("initialisation failed -> reduce radius or number of atoms")
            sys.exit()

    # remaining positive charges
    while i < n:
        x = random()*(dim[0]-radius) + radius
        y = random()*(dim[1]-radius) + radius
        angle = 2.0*random()*pi
        OVERLAP = 1
        for j in range(i):
            if dist(tmp_coord[j], [x, y]) < (1-OverlapFr)*2*radius:
                OVERLAP = 0
        if OVERLAP:
            tmp_coord.append([x, y, qat])
            tmp_orient.append(angle)
            i += 1
        ntrial += 1
        if ntrial > 100000:
            print("initialisation failed -> reduce radius or number of atoms")
            sys.exit()
    return tmp_coord, tmp_orient


Atom_Coord, Atom_Orient = InitConf(nAtoms, BoxDim, Radius, qat, frac_neg)
Color = [charge_color(a[2], qat) for a in Atom_Coord]
print(f"Placed {len(Atom_Coord)} atoms with random dipole orientations.")

## 7. Run the Monte Carlo

Run `nsteps` Metropolis trial moves (displacement / swap / rotation), recording the energy and
its components, the running acceptance ratio, the dipole-order parameter, and snapshots of the
positions **and** orientations.

In [ ]:
(traj, otraj, E_hist, Elj_hist, Ecoul_hist, Edip_hist,
 acc_hist, order_hist, move_hist) = run_mc(Atom_Coord, Atom_Orient)
print(f"Done: {len(traj)-1} MC steps. Final E = {E_hist[-1]:.1f}, "
      f"dipole order {order_hist[0]:.3f} -> {order_hist[-1]:.3f}")

## 8. Energy, dipole ordering and acceptance

**How the order parameter is computed.** As the dipoles order they line up **head-to-tail** with
their neighbours, so we measure a *local* alignment: the average of $\cos(\theta_i-\theta_j)$ over
all neighbour pairs closer than $1.5\,r_{min}$,

$$S = \big\langle \cos(\theta_i-\theta_j) \big\rangle_{\text{neighbour pairs}},$$

computed by `local_order`. $S\approx 0$ for random orientations and grows toward 1 as neighbouring
dipoles align.

**Why not the net polarization?** A tempting alternative — the length of the mean dipole vector,
$\big|\frac1N\sum_i(\cos\theta_i,\sin\theta_i)\big|$ — is *misleading here*: 2D dipoles order into
head-to-tail **chains and closed rings** that curl around, so the net polarization stays near
**zero even when the system is strongly ordered** (it can even *decrease* as ordering sets in). The
local measure $S$, together with the falling dipole energy $E_{dip}$, captures the ordering that the
net polarization hides.

The left panel below shows the total energy and its components — watch **$E_{dip}$ drop sharply** as
the dipoles bind head-to-tail (with `qat = 0` the Coulomb curve sits at zero). The right panel shows
the local order $S$ rising, together with the running acceptance ratio.

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 5))

steps = range(len(E_hist))
ax1.plot(steps, E_hist,     label="E$_{tot}$",   lw=1.5)
ax1.plot(steps, Elj_hist,   label="E$_{LJ}$",    lw=1.0)
ax1.plot(steps, Ecoul_hist, label="E$_{Coul}$",  lw=1.0)
ax1.plot(steps, Edip_hist,  label="E$_{dip}$",   lw=1.0)
ax1.set_xlabel("MC step")
ax1.set_ylabel("energy")
ax1.set_title("Energy along the Monte Carlo run")
ax1.legend()
ax1.grid(alpha=0.3)

ax2.plot(steps, order_hist, color="tab:purple", lw=1.4, label="local dipole order S")
ax2.set_xlabel("MC step")
ax2.set_ylabel("local dipole order  S", color="tab:purple")
ax2.tick_params(axis='y', labelcolor="tab:purple")
axb = ax2.twinx()
axb.plot(steps, acc_hist, color="tab:green", lw=1.0, alpha=0.7)
axb.set_ylabel("acceptance ratio", color="tab:green")
axb.set_ylim(0, 1)
axb.tick_params(axis='y', labelcolor="tab:green")
ax2.set_title(f"Local ordering (final S = {order_hist[-1]:.2f}, acceptance {acc_hist[-1]:.2f})")
ax2.grid(alpha=0.3)

plt.tight_layout()
plt.show()

## 9. Visualise the system

Start and end configurations side by side. Particles are **neutral** (grey) by default; the
**green arrows** are the dipole orientations. Watch the arrows go from random to locally aligned
(head-to-tail chains). (With `qat = Radius`, particles are coloured white = +, dark = −.)

In [ ]:
def draw_config(ax, coord, orient, title):
    ax.set_xlim(0, BoxDim[0])
    ax.set_ylim(0, BoxDim[1])
    ax.set_aspect('equal')
    ax.set_facecolor("#ccddff")
    ax.set_title(title)
    ax.invert_yaxis()   # match the original canvas (y downwards)
    for a in coord:
        ax.add_patch(Circle((a[0], a[1]), Radius, facecolor=charge_color(a[2], qat),
                            edgecolor="black", lw=0.8))
    X = [a[0] for a in coord]; Y = [a[1] for a in coord]
    U = [cos(o) for o in orient]; V = [sin(o) for o in orient]
    ax.quiver(X, Y, U, V, color="green", angles='xy', scale_units='xy',
              scale=1.0/(1.3*Radius), width=0.007, pivot='mid', zorder=5)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(11, 5.5))
draw_config(ax1, traj[0],  otraj[0],  f"Initial  (order = {order_hist[0]:.2f})")
draw_config(ax2, traj[-1], otraj[-1], f"Final  (order = {order_hist[-1]:.2f})")
plt.tight_layout()
plt.show()

## 10. Animation of the Monte Carlo run

Replays the whole trajectory — particles *and* their dipole arrows.
(To keep it light, only every few frames are shown — adjust `stride`.)

In [ ]:
stride = max(1, len(traj)//120)   # cap at ~120 frames
frames = list(range(0, len(traj), stride))

fig, ax = plt.subplots(figsize=(6, 6))
ax.set_xlim(0, BoxDim[0])
ax.set_ylim(0, BoxDim[1])
ax.set_aspect('equal')
ax.set_facecolor("#ccddff")
ax.invert_yaxis()

circles = [Circle((a[0], a[1]), Radius,
                  facecolor=charge_color(a[2], qat), edgecolor="black", lw=0.8)
           for a in traj[0]]
for c in circles:
    ax.add_patch(c)

X0 = [a[0] for a in traj[0]]; Y0 = [a[1] for a in traj[0]]
U0 = [cos(o) for o in otraj[0]]; V0 = [sin(o) for o in otraj[0]]
quiv = ax.quiver(X0, Y0, U0, V0, color="green", angles='xy', scale_units='xy',
                 scale=1.0/(1.3*Radius), width=0.007, pivot='mid', zorder=5)
title = ax.set_title("")

def update(frame_idx):
    f = frames[frame_idx]
    coord = traj[f]; orient = otraj[f]
    for c, a in zip(circles, coord):
        c.center = (a[0], a[1])
    quiv.set_offsets([(a[0], a[1]) for a in coord])
    quiv.set_UVC([cos(o) for o in orient], [sin(o) for o in orient])
    title.set_text(f"step {f}   E = {E_hist[f]:.0f}   order = {order_hist[f]:.2f}   ({move_hist[f]})")
    return circles + [quiv, title]

anim = FuncAnimation(fig, update, frames=len(frames), interval=80, blit=False)
plt.close(fig)   # avoid a duplicate static figure
anim

## 11. What the dipoles add

Compared with the plain `LJ-ELEC_MMC` notebook, the new ingredients are the **orientations**, the
**dipole–dipole energy**, and the **rotation move** — and they change the physics qualitatively:

* the dipole interaction is **anisotropic** (it depends on how the arrows point relative to the
  line joining two particles), so the system lowers its energy by ordering the dipoles into
  **head-to-tail chains and closed rings** — visible as the rising local order $S$ and the sharply
  falling $E_{dip}$;
* those chains and rings **curl around**, so the *global* net polarization stays near zero even
  though the dipoles are strongly ordered locally — a good reminder that the right order parameter
  depends on the kind of order you expect;
* orientations are a genuinely new set of degrees of freedom, sampled by the dedicated **rotation**
  move; without it the arrows would stay frozen at their random initial values;
* the particles are **neutral** by default so the ordering is purely dipolar; set `qat = Radius` to
  switch the monopole **Coulomb** interaction back on and watch it compete with (and partly
  scramble) the dipolar order;
* the dipole strength `mm` and the `Temperature` set the competition between dipolar ordering and
  thermal disorder — raise `mm` for stronger order (watch acceptance drop), or raise `Temperature`
  to melt it.

As in the other Monte Carlo notebooks, lowering `Temperature` slowly (*simulated annealing*) drives
the system toward its most ordered, lowest-energy state.